# Proyecto ML — Modelado y validación (Entrega 2)

**Objetivo de E2:** comparar 2–3 familias de modelos para clasificación de macro-género musical bajo un esquema de validación **sin leakage**, y seleccionar el modelo ganador con criterios estadísticos (no solo "el mayor accuracy").

**Conexión con E1:**
- Reutilizamos el mapeo de sub-géneros → macro-géneros (ver `src/genre_mapping.py`, justificación original en E1 sección 2).
- Reutilizamos la limpieza de datos (deduplicación por `track_id`, filtro de outliers básicos).
- **Corrección crítica:** detectamos en E2 que el split aleatorio de E1 producía *leakage por artista* (un mismo artista con tracks en train y test). E1 fue re-ejecutado con el split corregido; aquí usamos el mismo esquema desde el inicio. Ver `report/reporte_e2.md` sección "Corrección metodológica".

**Familias de modelos comparadas:**
1. **Lineal:** `LogisticRegression` multinomial (continuidad con E1).
2. **Tree ensemble — bagging:** `RandomForestClassifier`.
3. **Tree ensemble — boosting:** `HistGradientBoostingClassifier` (built-in sklearn, sin dependencias adicionales).

**Esquema de validación:**
- **Outer holdout (~20%):** `StratifiedGroupKFold(n_splits=5)`, primer fold como test. Agrupado por `artists` y estratificado por `macro_genre`. **No se toca hasta el final.**
- **Inner CV (sobre el 80% train):** `StratifiedGroupKFold(n_splits=5)` para producir distribución de scores por modelo → base para tests estadísticos.

**Secciones:**
0. Setup
1. Carga, limpieza y mapeo (reproducción de E1)
2. Estrategia de validación sin leakage
3. Pipeline de preprocesamiento
4. Familias de modelos a comparar
5. Cross-validation y captura de scores por fold *(pendiente — Slice 4)*
6. Comparación estadística entre modelos *(pendiente — Slice 5)*
7. Evaluación final en test holdout *(pendiente — Slice 5)*
8. Conclusiones *(pendiente — Slice 6)*

## 0. Setup

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.genre_mapping import (
    CATEGORICAL_FEATURES,
    GENRE_MAPPING,
    MACRO_GENRES,
    NUMERIC_FEATURES,
)

RANDOM_STATE = 42
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "spotify_tracks.csv"

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"DATA_PATH:    {DATA_PATH}")
print(f"Existe CSV?:  {DATA_PATH.exists()}")
print(f"Features numéricas:    {NUMERIC_FEATURES}")
print(f"Features categóricas:  {CATEGORICAL_FEATURES}")
print(f"Macro-géneros target: {MACRO_GENRES}")

PROJECT_ROOT: D:\Programming\EAFIT\ML\Proyecto_ML
DATA_PATH:    D:\Programming\EAFIT\ML\Proyecto_ML\data\raw\spotify_tracks.csv
Existe CSV?:  True
Features numéricas:    ['danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms']
Features categóricas:  ['key', 'mode', 'time_signature', 'explicit']
Macro-géneros target: ['ambient', 'asian-pop', 'classical', 'electronic', 'folk', 'hip-hop', 'jazz', 'kids-comedy', 'latin', 'metal', 'other', 'pop', 'reggae', 'rock', 'soul-funk', 'world']


## 1. Carga, limpieza y mapeo de géneros

Reproducimos el flujo de E1 para tener un punto de partida consistente:

1. Cargar `spotify_tracks.csv`.
2. Eliminar columnas índice residuales (`Unnamed: 0`).
3. Deduplicar por `track_id` (la misma canción puede aparecer en múltiples sub-géneros).
4. Filtrar outliers básicos: `duration_ms < 30 s` o `tempo == 0`.
5. Aplicar `GENRE_MAPPING` y rellenar con `"other"` los sub-géneros no mapeados.

In [2]:
df = pd.read_csv(DATA_PATH)
df = df.drop(columns=[c for c in df.columns if c.startswith("Unnamed")], errors="ignore")

n_raw = len(df)
df = df.drop_duplicates(subset="track_id").reset_index(drop=True)
n_dedup = len(df)

mask_bad = (df["duration_ms"] < 30_000) | (df["tempo"] == 0)
df = df.loc[~mask_bad].reset_index(drop=True)
n_clean = len(df)

df["macro_genre"] = df["track_genre"].map(GENRE_MAPPING).fillna("other")

print(f"Filas crudas:                   {n_raw:,}")
print(f"Tras deduplicar por track_id:   {n_dedup:,}  (-{n_raw - n_dedup:,})")
print(f"Tras filtrar outliers básicos:  {n_clean:,}  (-{n_dedup - n_clean:,})")
print(f"\nDistribución de macro-géneros:")
print(df["macro_genre"].value_counts())

Filas crudas:                   114,000
Tras deduplicar por track_id:   89,741  (-24,259)
Tras filtrar outliers básicos:  89,571  (-170)

Distribución de macro-géneros:
macro_genre
electronic     14890
rock            9661
latin           9176
world           8774
metal           7426
folk            7113
asian-pop       6270
kids-comedy     4847
ambient         4798
pop             3561
soul-funk       3448
reggae          2874
classical       2508
hip-hop         2235
other           1467
jazz             523
Name: count, dtype: int64


## 2. Estrategia de validación sin leakage

**Problema detectado en E1:** un split aleatorio (`train_test_split` estratificado solo por clase) permitía que canciones de un mismo artista cayeran en train y test. Con 11.265 artistas que tienen ≥2 tracks (top: George Jones con 260, The Beatles con 149), el modelo aprendía la *firma sonora del artista* en lugar del género real, inflando las métricas reportadas.

**Solución adoptada (E2):**

- **Outer holdout (≈20% test, ≈80% train):** `StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)` tomando el **primer fold** como test. Esto garantiza:
    - Estratificación por `macro_genre` (clases desbalanceadas).
    - Grupos por `artists`: ningún artista aparece en train y test simultáneamente.
    - Test set **intocado** hasta la evaluación final.
- **Inner CV (sobre el train):** `StratifiedGroupKFold(n_splits=5)` para producir 5 estimaciones de cada métrica por modelo. Sin esto no se pueden hacer tests estadísticos pareados.

**Métrica primaria:** `macro_f1` (clases desbalanceadas).  
**Secundarias:** `accuracy`, `balanced_accuracy`, `top_3_accuracy`.

In [3]:
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
TARGET = "macro_genre"
GROUP = "artists"

X = df[FEATURES].copy()
y = df[TARGET].copy()
groups = df[GROUP].copy()

# Outer holdout: primer fold de un StratifiedGroupKFold(n_splits=5) → ~20% test
outer_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
train_idx, test_idx = next(outer_cv.split(X, y, groups=groups))

X_train = X.iloc[train_idx].reset_index(drop=True)
X_test = X.iloc[test_idx].reset_index(drop=True)
y_train = y.iloc[train_idx].reset_index(drop=True)
y_test = y.iloc[test_idx].reset_index(drop=True)
groups_train = groups.iloc[train_idx].reset_index(drop=True)
groups_test = groups.iloc[test_idx].reset_index(drop=True)

print(f"Train: {X_train.shape}  |  artistas: {groups_train.nunique():,}")
print(f"Test:  {X_test.shape}  |  artistas: {groups_test.nunique():,}")
overlap = len(set(groups_train) & set(groups_test))
print(f"Solapamiento de artistas train ∩ test: {overlap}  (debe ser 0)")

print(f"\nDistribución de clases en train (%):")
print((y_train.value_counts(normalize=True) * 100).round(2))
print(f"\nDistribución de clases en test (%):")
print((y_test.value_counts(normalize=True) * 100).round(2))

Train: (71657, 14)  |  artistas: 25,100
Test:  (17914, 14)  |  artistas: 6,286
Solapamiento de artistas train ∩ test: 0  (debe ser 0)

Distribución de clases en train (%):
macro_genre
electronic     16.62
rock           10.79
latin          10.24
world           9.80
metal           8.29
folk            7.94
asian-pop       7.00
kids-comedy     5.41
ambient         5.36
pop             3.97
soul-funk       3.85
reggae          3.21
classical       2.80
hip-hop         2.50
other           1.64
jazz            0.59
Name: proportion, dtype: float64

Distribución de clases en test (%):
macro_genre
electronic     16.62
rock           10.78
latin          10.24
world           9.80
metal           8.29
folk            7.94
asian-pop       7.00
kids-comedy     5.41
ambient         5.36
pop             3.98
soul-funk       3.85
reggae          3.21
classical       2.80
hip-hop         2.50
other           1.64
jazz            0.57
Name: proportion, dtype: float64


### 2.1 CV interno para comparar modelos

Definimos un `StratifiedGroupKFold` con 5 folds sobre el `train` para que cada modelo produzca **5 scores** por métrica. Estos scores son la base de:

- Intervalos de confianza por bootstrap.
- Paired t-test / Wilcoxon signed-rank entre pares de modelos.
- Tamaño de efecto (Cohen's d).
- Corrección por comparaciones múltiples (Bonferroni).

In [4]:
inner_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Verificación rápida: los 5 folds del inner CV tampoco deben tener leakage de artista.
for fold, (tr_idx, va_idx) in enumerate(
    inner_cv.split(X_train, y_train, groups=groups_train), 1
):
    g_tr = set(groups_train.iloc[tr_idx])
    g_va = set(groups_train.iloc[va_idx])
    overlap = len(g_tr & g_va)
    print(
        f"Fold {fold}: train={len(tr_idx):>6,}  val={len(va_idx):>5,}  "
        f"artistas val={len(g_va):>5,}  solapamiento={overlap}"
    )

Fold 1: train=57,337  val=14,320  artistas val=5,031  solapamiento=0
Fold 2: train=57,307  val=14,350  artistas val=5,002  solapamiento=0
Fold 3: train=57,337  val=14,320  artistas val=5,027  solapamiento=0
Fold 4: train=57,335  val=14,322  artistas val=5,039  solapamiento=0
Fold 5: train=57,312  val=14,345  artistas val=5,001  solapamiento=0


## 3. Pipeline de preprocesamiento

`ColumnTransformer` aplica transformaciones distintas a numéricas vs categóricas, y se mete dentro de un `Pipeline` con cada modelo. Así `scikit-learn` re-fittea el `StandardScaler` y el `OneHotEncoder` **solo sobre el train de cada fold** durante CV — evita leakage de transformaciones.

In [5]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), NUMERIC_FEATURES),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            CATEGORICAL_FEATURES,
        ),
    ]
)
preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. `

## 4. Familias de modelos a comparar

Definimos las tres familias dentro de `Pipeline`. Hiperparámetros razonables (no tuneados aún — el tuning extensivo va en E3).

| Modelo | Familia | Hiperparámetros clave |
|---|---|---|
| `LogisticRegression` | Lineal | `multi_class='multinomial'`, `C=1.0`, `max_iter=2000` |
| `RandomForestClassifier` | Tree ensemble (bagging) | `n_estimators=300`, `max_depth=None`, `n_jobs=-1` |
| `HistGradientBoostingClassifier` | Tree ensemble (boosting) | `max_iter=200`, defaults |

**Notas:**
- LogReg y RF necesitan preprocesamiento (escala + one-hot). HGB es robusto a escala y puede manejar categóricas numéricas directamente, pero por consistencia usamos el mismo preprocessor.
- HGB elegido sobre XGBoost/LightGBM para mantener `requirements.txt` sin dependencias nuevas; el desempeño es comparable y está built-in en sklearn.

In [6]:
MODELS: dict[str, Pipeline] = {
    "logreg": Pipeline(
        steps=[
            ("pre", preprocessor),
            (
                "clf",
                LogisticRegression(
                    max_iter=2000,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    ),
    "random_forest": Pipeline(
        steps=[
            ("pre", preprocessor),
            (
                "clf",
                RandomForestClassifier(
                    n_estimators=300,
                    max_depth=None,
                    n_jobs=-1,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    ),
    "hist_gb": Pipeline(
        steps=[
            ("pre", preprocessor),
            (
                "clf",
                HistGradientBoostingClassifier(
                    max_iter=200,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    ),
}

for name, pipe in MODELS.items():
    print(f"  {name:<14}  →  {pipe.named_steps['clf'].__class__.__name__}")

  logreg          →  LogisticRegression
  random_forest   →  RandomForestClassifier
  hist_gb         →  HistGradientBoostingClassifier


## 5. Cross-validation y captura de scores por fold

*Pendiente (Slice 4)* — entrenar cada modelo con `cross_validate` sobre el inner CV, capturar todas las métricas por fold y guardar para los tests estadísticos.

## 6. Comparación estadística entre modelos

*Pendiente (Slice 5)* — paired t-test, Wilcoxon signed-rank, Cohen's d, corrección Bonferroni, intervalos de confianza por bootstrap.

## 7. Evaluación final en test holdout

*Pendiente (Slice 5)* — entrenar el modelo ganador en todo el train y evaluar **una sola vez** en el test holdout intocado.

## 8. Conclusiones

*Pendiente (Slice 6)* — redactar hallazgos, comparar con E1 corregido, definir modelo para E3.